<a href="https://colab.research.google.com/github/JianfengMI/MLprojects/blob/main/PEAD_scanner.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Post Earnings Announcement Drift (PEAD)

Rational: to catch the momentum of extraordinary jump on earnings by precise entry timing.

Steps:
1. initial stock identification

    1.1 price change >=10%

    1.2 volume >= 2 x 50DMA

    1.3 market cap > $100M

    1.4 price > $1

2. the consolidation analysis (after step 1)

    2.1 correction depth < 25%

    2.2 look out for  red days (red candles < average daily range)

    2.3 look out for MA support

    2.4 minimum consolidation period > 1 week

    2.5 volatility contraction

    2.6 low risk entry point

3. Exit PEAD trades

    similar to Kristjan Qullamagi's strategy

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import yfinance as yf
from datetime import datetime, timedelta
import math
from tqdm.auto import tqdm
import requests
import matplotlib.pyplot as plt

In [ ]:
# set parameters
START_DATE = (datetime.today() - timedelta(days=365*20)).strftime('%Y-%m-%d') # 5y of history // change to 20 years
END_DATE = datetime.today().strftime('%Y-%m-%d')

# download S&P500 tickers and their prices
def get_sp500_tickers():
    url = "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies"
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
    }
    response = requests.get(url, headers=headers)
    response.raise_for_status() # Raise an exception for HTTP errors

    # pandas can directly parse tables from HTML content
    tables = pd.read_html(response.text)

    df = None
    # Iterate through found tables to identify the S%P 500 constituents table
    for df_candidate in tables:
        # Check for columns commonly present in the S%P 500 constituents table
        # like 'Symbol', 'Ticker', or 'symbol' (case-insensitive)
        # Convert column names to string before comparison
        cols = [c for c in df_candidate.columns if "Symbol" in str(c) or "Ticker" in str(c) or "symbol" in str(c).lower()]
        if cols:
            df = df_candidate
            break # Found the table

    if df is None:
        raise ValueError("Could not find S%26P 500 constituents table on the Wikipedia page.")

    # Ensure 'Symbol' or 'Ticker' column is correctly identified
    # Convert column names to string before comparison
    col_name = [c for c in df.columns if "Symbol" in str(c) or "Ticker" in str(c) or "symbol" in str(c).lower()][0]

    # Clean up ticker symbols (e.g., BRK.B -> BRK-B)
    tickers = df[col_name].astype(str).str.replace(".", "-", regex=False).tolist()
    return tickers

def download_stock_prices(tickers_to_download, start, end, batch_size=80):
    """
    Download full OHLCV price data in batches using yfinance.
    Returns a MultiIndex DataFrame:
        columns = (ticker, price_field)
        index   = dates
    """
    all_data = pd.DataFrame()
    tickers_list = list(tickers_to_download)

    for i in tqdm(range(0, len(tickers_list), batch_size), desc="Downloading prices"):
        batch = tickers_list[i:i+batch_size]

        try:
            data = yf.download(
                batch, start=start, end=end,
                progress=False, group_by='ticker', auto_adjust=True
            )

            if data.empty:
                print(f"Skipping batch {batch}: No data downloaded.")
                continue

            # If single ticker, YF returns normal DataFrame → convert it to MultiIndex
            if not isinstance(data.columns, pd.MultiIndex):
                t = batch[0]
                data.columns = pd.MultiIndex.from_product([[t], data.columns])

            # Merge batch with accumulated data
            if all_data.empty:
                all_data = data
            else:
                all_data = pd.concat([all_data, data], axis=1)

        except Exception as e:
            print(f"Batch download error for {batch}: {e}")

    # Ensure sorted index and columns
    if not all_data.empty:
        all_data = all_data.sort_index()
        all_data = all_data.sort_index(axis=1)

    return all_data

# Step 1. download the stock price and find the event signals

In [ ]:
# Define MIN_PRICE_HISTORY_DAYS before it's used
MIN_PRICE_HISTORY_DAYS = 252 # Approximately 1 year of trading days

# Fetch tickers and prices
sp500_tickers = get_sp500_tickers()
universe = [t for t in sp500_tickers]  # copy

prices = download_stock_prices(universe, start=START_DATE, end=END_DATE)
# Remove tickers with insufficient history
valid = [t for t in sp500_tickers if prices.get(t, pd.Series()).dropna().shape[0] >= MIN_PRICE_HISTORY_DAYS]
print(f"{len(valid)} tickers with >= {MIN_PRICE_HISTORY_DAYS} days of data.")

In [ ]:
data = prices.copy()

In [ ]:
# mount on google drive and save the data
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# save the prices data
prices.to_csv('/content/drive/MyDrive/ML/prices_20y.csv')

In [ ]:
data = pd.read_csv('/content/drive/MyDrive/ML/prices_20y.csv', header=[0,1], index_col=0)
data.index = pd.to_datetime(data.index)

In [ ]:
# prepare S&P 500 index data, don't keet ticker's name

spx = yf.download("^GSPC", start=START_DATE, end=END_DATE)

spx.columns = spx.columns.droplevel(1)

spx

def label_regime(spy):

    close = spy["Close"]
    ma200 = close.rolling(200).mean()

    ret3 = close.pct_change(63)

    regime = []

    for i in range(len(close)):

        if close.iloc[i] > ma200.iloc[i] and ret3.iloc[i] > 0:
            regime.append(1)   # bull
        elif close.iloc[i] < ma200.iloc[i]:
            regime.append(-1)  # bear
        else:
            regime.append(0)   # neutral

    spy["regime"] = regime

    return spy

spx = label_regime(spx)

# combine data with spx['regime']
# First, align the spx['regime'] Series to the 'data' DataFrame's index.
# Using 'ffill' will propagate the last valid observation forward to fill any missing dates.
aligned_regime = spx['regime'].reindex(data.index, method='ffill')

# Iterate through each unique ticker and add the aligned 'regime' series
# as a new sub-column under that ticker in the MultiIndex DataFrame.
for ticker in data.columns.get_level_values(0).unique():
    data[ticker, 'regime'] = aligned_regime

In [ ]:
# earnings event detection
def detect_event(price):

    gap = (
        price["Open"].iloc[-1] -
        price["Close"].iloc[-2]
    ) / price["Close"].iloc[-2]

    vol_ratio = (
        price["Volume"].iloc[-1] /
        price["Volume"].iloc[-50:].mean()
    )

    if abs(gap) > 0.1 and vol_ratio > 2:
        return True, gap, vol_ratio # Return all 3 values when True

    return False, gap, vol_ratio # Return all 3 values even when False

In [ ]:
# Earnings jump scanner
def advanced_earnings_scan(df,lookback_days):

    signals = []

    # Get the actual unique tickers present in the first level of the MultiIndex
    actual_tickers = df.columns.get_level_values(0).unique()

    for ticker in actual_tickers:

        price = df[ticker].dropna()

        if len(price) < 150:
            continue

        # check last N days
        for i in range(lookback_days, 0, -1):
             hist = price.iloc[:-i]

             # Ensure 'hist' has enough data points for detect_event
             if len(hist) < 2:
                 continue

            # Event detection
             event_ok, gap, vol_ratio = detect_event(hist)

             if not event_ok:
                continue

             event_price = hist["Close"].iloc[-1]
             event_date = hist.index[-1]
             event_volume = hist["Volume"].iloc[-1]
             event_jump = gap

             # Calculate
             regime = hist["regime"].iloc[-1]

             signals.append({
                "Ticker": ticker,
                "Event_Date": event_date,
                "Event_Price": event_price,
                "Event_Volume": event_volume,
                "Event_Jump": event_jump,
                "regime": regime
            })

    signals = pd.DataFrame(signals)

    if len(signals) == 0:
        # print("No earnings events in the last", lookback_days, "days")
        return signals

    signals = signals.sort_values("Event_Date", ascending=False)

    return signals

signals = advanced_earnings_scan(data, len(data))

In [ ]:
len(signals)

In [ ]:
earnings_signals = signals.copy()

In [ ]:
# save earnings signals on google drive
earnings_signals.to_csv('/content/drive/MyDrive/ML/earnings_signals.csv')

In [ ]:
# download earnings signals from google drive
signals = pd.read_csv('/content/drive/MyDrive/ML/earnings_signals.csv', index_col=0)

In [ ]:
calendar = pd.DataFrame()
for ticker_symbol in signals["Ticker"].unique():
  ticker = yf.Ticker(ticker_symbol)

  # Get earnings dates
  earnings = ticker.get_earnings_dates(limit=80)

  # Check if earnings DataFrame is not empty before processing
  if not earnings.empty:
    # Assign the index (which contains the datetime) to a new 'date' column
    earnings['date'] = earnings.index
    # Convert the 'date' column to contain only the date part
    earnings['date'] = earnings['date'].dt.date
    earnings['Ticker'] = ticker_symbol

    calendar = pd.concat([calendar, earnings], ignore_index=True)

In [ ]:
# save calendar on google drive
calendar.to_csv('/content/drive/MyDrive/ML/calendar.csv')

In [ ]:
# download calendar data from google drive
calendar = pd.read_csv('/content/drive/MyDrive/ML/calendar.csv', index_col=0)

In [ ]:
len(calendar)

In [ ]:
def match_earnings(jump_date, earnings_date):
  return jump_date == earnings_date or jump_date == earnings_date + pd.Timedelta(days=1)

In [ ]:
signals['Event_Date'] = pd.to_datetime(signals['Event_Date'])
calendar['date'] = pd.to_datetime(calendar['date'])

In [ ]:
# better method, avoid the O(N x M) nested loops, use merge_asof
true_signals = pd.merge_asof(
    signals.sort_values('Event_Date'),
    calendar.sort_values('date'),
    left_on='Event_Date',
    right_on='date',
    by='Ticker',
    direction='nearest',
    tolerance=pd.Timedelta('1D')
)

In [ ]:
true_signals

In [ ]:
len(true_signals)

In [ ]:
# drop off NaN data on date
true_signals = true_signals.dropna(subset=['date'])

In [ ]:
true_signals

# Step 2. the consolidation analysis

Earnings Jump → Pullback → Tight Consolidation → Breakout Entry

In [ ]:
def check_consolidation(price, event_date):

    post_df = price.loc[event_date : event_date + pd.Timedelta(days=10)].copy() # check following 2 weeks after earnings

    if len(post_df) < 5:
        return 0

    score = 0

    # Rule 1
    peak = post_df['High'].iloc[0]
    trough = post_df['Low'].min()
    if (peak - trough) / peak <= 0.25:
        score += 1

    # Rule 2
    post_df['range'] = post_df['High'] - post_df['Low']
    avg_range = post_df['range'].mean()
    red_days = post_df[post_df['Close'] < post_df['Open']]
    if len(red_days) > 0 and (red_days['range'] < avg_range).mean() > 0.7:
        score += 1

    # Rule 3
    ma20 = price['Close'].rolling(20).mean()
    post_df['ma20'] = ma20.loc[post_df.index]
    if (post_df['Low'] > post_df['ma20'] * 0.98).mean() > 0.7:
        score += 1

    # Rule 4
    valid_length = len(post_df)
    if valid_length >= 5:
      score += 1

    # Rule 5 (vol contraction)
    vol = post_df['range'] / post_df['Close']
    if vol.iloc[-3:].mean() <= vol.mean():
        score += 1

    # Rule 6 (tight near highs)
    total_range = (post_df['High'].max() - post_df['Low'].min()) / post_df['Close'].iloc[-1]
    near_high = post_df['Close'].iloc[-1] > post_df['High'].max() * 0.9

    if total_range < 0.1 and near_high:
        score += 1

    return score

In [ ]:
# based on score system
def find_entry(price, event_date, lookforward=10, max_entry_days=15):
    price = price.sort_index()

    if event_date not in price.index:
        return None, 0

    event_idx = price.index.get_loc(event_date)

    # consolidation window
    post_df = price.iloc[event_idx : event_idx + lookforward]

    if len(post_df) < 5:
        return None, 0

    # softer pivot
    pivot = post_df['High'].rolling(3).max().iloc[-1]

    # entry window
    future_df = price.iloc[event_idx + lookforward : event_idx + max_entry_days].copy()

    if len(future_df) == 0:
        return None, 0

    future_df['vol_avg'] = future_df['Volume'].rolling(20).mean()

    # scoring system
    for i in range(len(future_df)):
        row = future_df.iloc[i]

        score = 0

        # Rule 1: intraday breakout (looser)
        if row['High'] > pivot:
            score += 2

        # Rule 2: close strength
        if row['Close'] > pivot * 0.98:
            score += 1

        # Rule 3: volume confirmation (relaxed)
        if row['Volume'] > 1.2 * row['vol_avg']:
            score += 1

        # Rule 4: strong close in range
        if (row['Close'] - row['Low']) / (row['High'] - row['Low'] + 1e-6) > 0.6:
            score += 1

        # Entry threshold
        if score >= 3:
            return future_df.index[i], score

    return None, 0

In [ ]:
true_signals['Event_Date'] = pd.to_datetime(true_signals['Event_Date'])

true_signals['Consolidation_Score'] = 0
true_signals['Entry_Date'] = None

grouped = true_signals.groupby('Ticker')

for ticker, group in grouped:
    price = data.get(ticker)

    if price is None:
        continue

    price = price.sort_index()

    for idx, row in group.iterrows():
        event_date = row['Event_Date']

        try:
            score = check_consolidation(price, event_date)
            entry, signal_score = find_entry(price, event_date)
        except:
            score = 0
            entry = None
            signal_score = 0

        true_signals.loc[idx, 'Consolidation_Score'] = score
        true_signals.loc[idx, 'Entry_Date'] = entry.date() if entry is not None else None
        true_signals.loc[idx, 'Entry_Price'] = price.loc[entry, 'Close'] if entry is not None else None
        true_signals.loc[idx, 'Signal_Score'] = signal_score

In [ ]:
entrys = true_signals.dropna(subset=['Entry_Date'])

In [ ]:
entrys.columns

In [ ]:
entrys['Action'] = np.where(entrys['Surprise(%)'] > 0, 1, -1)

In [ ]:
# arbitarily set Risk at 0.08
entrys['Stop'] = np.where(entrys['Action'] ==1, entrys['Entry_Price'] * (1 - 0.08), entrys['Entry_Price'] * (1 + 0.08))

# Step 3. backtest the entries

In [ ]:
def calculate_profit(price, action, entry_date, entry_price, stop_price, max_holding_days=40):
  """
    Returns:
        profit,
        lowest price,
        highest price,
        days_to_peak,
        days_to_bottom,
        risk
    """

  # 0. start after entry day
  trade = price.loc[entry_date:].iloc[1:].copy()

  if len(trade) <= max_holding_days:
    return 0,0,0,0,0,0

  # 1. Apply holding cap
  trade = trade.iloc[:max_holding_days]

  # 2. Find lowest / highest price and its date
  max_price = trade["Close"].max()
  max_idx = trade["Close"].idxmax()

  min_price = trade["Close"].min()
  min_idx = trade["Close"].idxmin()

  # 4. Compute days to peak
  days_to_peak = trade.index.get_loc(max_idx) + 1
  days_to_bottom = trade.index.get_loc(min_idx) + 1

  # Profit
  if action > 0:
    profit = (max_price - entry_price) / entry_price
    # Initialize risk with the stop-loss distance
    risk = abs(stop_price - entry_price) / entry_price
    # If the price dropped further than the stop before the peak, that's the actual risk faced
    if min_price < stop_price and days_to_bottom < days_to_peak:
        profit = (stop_price - entry_price) / entry_price
        risk = abs(min_price - entry_price) / entry_price

  elif action < 0:
    profit = (entry_price - min_price) / entry_price
    # Initialize risk with the stop-loss distance
    risk = abs(stop_price - entry_price) / entry_price
    # If the price rose further than the stop (for short) before the bottom, that's the actual risk faced
    if max_price > stop_price and days_to_bottom > days_to_peak:
        profit = (entry_price - stop_price) / entry_price
        risk = abs(max_price - entry_price) / entry_price

  return profit, risk, days_to_peak, days_to_bottom, max_price, min_price

In [ ]:
entrys.head()

In [ ]:
results = []

for _, row in entrys.iterrows():

    ticker = row["Ticker"]
    entry_date = row["Entry_Date"]
    entry_price = row["Entry_Price"]
    stop_price = row["Stop"]
    action = row["Action"]

    price = data[ticker].dropna()

    profit, risk, days_to_peak, days_to_bottom, max_price, min_price = calculate_profit(
        price,
        action,
        entry_date,
        entry_price,
        stop_price,
        max_holding_days=40
    )

    results.append({
        **row,
        "Profit": round(profit, 4),
        "Risk": round(risk, 4),
        "Days_to_Peak": days_to_peak,
        "Days_to_Bottom": days_to_bottom,
        "lowest_price": min_price,
        "highest_price": max_price,
        # "Label
        # "Label": label,
        # "Time_to_Outcome": time_to_outcome
    })

results = pd.DataFrame(results)

In [ ]:
results.columns

In [ ]:
# does signal score correlate with profit
results[["Signal_Score", "Profit"]].corr()

In [ ]:
# does surprise correlate with profit
results[["Surprise(%)", "Profit"]].corr()

In [ ]:
# how about Consolidation_Score
results[["Consolidation_Score", "Profit"]].corr()

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# Make the plot nicer and more readable
plt.figure(figsize=(10, 7))

sns.scatterplot(
    data=results,
    x='Profit',
    y='Surprise(%)',
    hue='Action',
    palette={1: 'green', -1: 'red'}
)

# Improve labels and title
plt.title('Surprise(%) Distribution by Profit', fontsize=14, pad=20)
plt.xlabel('Profit', fontsize=12)
plt.ylabel('Surprise(%)', fontsize=12)
plt.ylim([-100,100])

plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# does there is a threshold to increase the win rate?
# first to see the average or median Signal_Score
results.groupby("Action")["Signal_Score"].mean()

In [ ]:
results.groupby("Action")["Signal_Score"].median()

In [ ]:
# second to see the win rate for each action
results.groupby("Action")["Profit"].apply(lambda x: (x > 0).mean())

In [ ]:
results.groupby("Action")["Profit"].sum()

In [ ]:
results[results["Profit"] > 0]["Risk"].mean()

In [ ]:
results[results["Profit"] > 0]["Risk"].median()

In [ ]:
results["Risk"].describe()

In [ ]:
results[results['Profit'] > 0]["Risk"].describe()

In [ ]:
profit_risk = results[(results['Risk'] < 0.09) & (results['Profit'] > 0 )].groupby("Action")["Profit"].sum()
loss_risk = results[(results['Risk'] < 0.09) & (results['Profit'] < 0 )].groupby("Action")["Profit"].sum()

In [ ]:
profit_loss_risk = pd.DataFrame([profit_risk, loss_risk]).T
profit_loss_risk.columns = ['Profit', 'Loss']
profit_loss_risk['ratio'] = abs(profit_loss_risk['Profit'] / profit_loss_risk['Loss'])

In [ ]:
profit_loss_risk

In [ ]:
# Action × Profit Distribution
results.boxplot(column="Profit", by="Action")

In [ ]:
results[results['Risk'] <= 0.08].boxplot(column="Profit", by="Action")

In [ ]:
# Expectancy
win = results[results["Profit"] > 0].groupby("Action")["Profit"].mean()
loss = results[results["Profit"] <= 0].groupby("Action")["Profit"].mean()
# Calculate win_rate as the proportion of trades with profit > 0 for each action
win_rate = results.groupby("Action")["Profit"].apply(lambda x: (x > 0).mean())

expectancy = win_rate * win + (1 - win_rate) * loss

In [ ]:
expectancy

In [ ]:
# if using 0.081 as Risk cut, what is the average days to the maximum price
results[results['Risk'] < 0.081]['Days_to_Peak'].mean(), results[results['Risk'] < 0.081]['Days_to_Peak'].median()

In [ ]:
results[results['Risk'] < 0.081]['Days_to_Bottom'].mean(), results[results['Risk'] < 0.081]['Days_to_Bottom'].median()

In [ ]:
results['Profit'].sum()

In [ ]:
year_2024 = results[(results['Entry_Date'] >= pd.to_datetime('2024-01-01').date()) & (results['Entry_Date'] < pd.to_datetime('2025-01-01').date())]

In [ ]:
year_2024.sort_values('Profit', ascending=False)['Profit'].plot(kind='bar')

In [ ]:
year_2025 = results[results['Entry_Date'] >= pd.to_datetime('2025-01-01').date()]

In [ ]:
year_2025.sort_values('Profit', ascending=False)['Profit'].plot(kind='bar')

In [ ]:
year_2026 = results[results['Entry_Date'] >= pd.to_datetime('2026-01-01').date()]

In [ ]:
year_2026.sort_values('Profit', ascending=False)['Profit'].plot(kind='bar')

In [ ]:
results['Risk'].describe()

In [ ]:
# what is the RR rato
results['RR'] = results['Profit'] / results['Risk']

In [ ]:
results['RR'].mean(), results['RR'].median()

In [ ]:
results['RR'].sort_values().plot(kind='bar')

In [ ]:
# if set the Risk at 0.081
risk_screened = results[results['Risk'] < 0.081]

In [ ]:
risk_screened['RR'].mean(), risk_screened['RR'].median()

In [ ]:
risk_screened['RR'].sort_values().plot(kind='bar')

In [ ]:
year_2025 = risk_screened[risk_screened['Entry_Date'] >= pd.to_datetime('2025-01-01').date()]

In [ ]:
year_2025['RR'].sort_values().plot(kind='bar')

In [ ]:
year_2025['RR'].mean(), year_2025['RR'].median()

In [ ]:
results.columns

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import matplotlib.ticker as mticker

def plot_trade_full(
    df,
    results,
    ticker,
    start_date=None,
    end_date=None,
    entry_date=None,
    entry_price=None,
    stop_price=None
):

    price = df[ticker].dropna().copy()

    results['Entry_Date'] = pd.to_datetime(results['Entry_Date'])


    # --- Apply window ---
    if start_date is not None:
        price = price.loc[pd.to_datetime(start_date):]

    if end_date is not None:
        price = price.loc[:pd.to_datetime(end_date)]

    if len(price) == 0:
        print("No data in selected window.")
        return

    # --- Pull trade info ONLY if not provided ---
    if entry_date is None:
        filtered = results[results["Ticker"] == ticker].copy()

        if start_date is not None:
            filtered = filtered[filtered["Entry_Date"] >= pd.to_datetime(start_date)]

        if end_date is not None:
            filtered = filtered[filtered["Entry_Date"] <= pd.to_datetime(end_date)]

        if len(filtered) == 0:
            print("No signals in selected window.")
            return

        # pick the most recent signal INSIDE window
        row = filtered.sort_values("Entry_Date").iloc[0]  # changed from -1

        entry_date = row["Entry_Date"]
        entry_price = row["Entry_Price"]
        stop_price = row["Stop"]

    entry_date = pd.to_datetime(entry_date)

    # --- Moving averages ---
    price["MA10"] = price["Close"].rolling(10).mean()
    price["MA20"] = price["Close"].rolling(20).mean()
    price["MA50"] = price["Close"].rolling(50).mean()

    fig, ax = plt.subplots(figsize=(14, 7))

    ax.plot(price.index, price["Close"], label="Close", linewidth=2)
    ax.plot(price.index, price["MA10"], label="MA10")
    ax.plot(price.index, price["MA20"], label="MA20")
    ax.plot(price.index, price["MA50"], label="MA50")

    # =========================
    #  ENTRY MARKER (robust)
    # =========================
    if entry_date is not None and entry_price is not None:

        entry_date = pd.to_datetime(entry_date)

        if entry_date < price.index.min() or entry_date > price.index.max():
            print(f"Entry date {entry_date} OUTSIDE plot range")
        else:
            pos = price.index.searchsorted(entry_date)

            if pos >= len(price.index):
                print("Entry position out of bounds")
            else:
                entry_idx = price.index[pos]

                print(f"Plotting entry at {entry_idx}, price={entry_price}")

                ax.scatter(
                    entry_idx,
                    entry_price,
                    marker="^",
                    s=200,
                    zorder=10,
                    label="Entry",
                    edgecolors="black"
                )

            # =========================
            #  STOP MARKER (after entry only)
            # =========================
            if stop_price is not None:
                future = price.loc[entry_idx:]

                stopped = future["Low"] <= stop_price

                if stopped.any():
                    stop_idx = stopped.idxmax()

                    ax.scatter(
                        stop_idx,
                        stop_price,
                        marker="x",
                        s=150,
                        zorder=5,
                        label="Stop Hit"
                    )

    # --- Guide lines ---
    if entry_price is not None:
        ax.axhline(entry_price, linestyle="--", alpha=0.3)

    if stop_price is not None:
        ax.axhline(stop_price, linestyle="--", alpha=0.3)

    # --- X-axis control (BEST PRACTICE) ---
    ax.xaxis.set_major_locator(mticker.MaxNLocator(8))

    plt.xticks(rotation=45)

    ax.set_title(f"{ticker} Trade View\nEntry: {entry_date} @ {entry_price}")
    ax.set_xlabel("Date")
    ax.set_ylabel("Price")

    ax.legend()
    plt.tight_layout()
    plt.show()

In [ ]:
# Ensure data index is datetime before plotting
data.index = pd.to_datetime(data.index)

In [ ]:
results[results['Action'] == -1].sort_values("Profit", ascending=False)

In [ ]:
plot_trade_full(
    data,
    results,
    ticker="CIEN",
    start_date="2011-06-01",
    end_date="2011-08-26"
)

## Random choose test

start with $10,000. randomly chose a entry and exit as mentioned in results. To see the real-time performance

In [ ]:
import pandas as pd
import random
from datetime import timedelta

def investment_random_near(df, initial=10000, max_holding_days=40, window_days=5):
    df = df.sort_values("Entry_Date").reset_index(drop=True)

    capital = initial
    next_available_date = df['Entry_Date'].min()

    results = []

    while True:
        # Step 1: all future trades
        future_trades = df[df['Entry_Date'] >= next_available_date]

        if future_trades.empty:
            break

        # Step 2: define time window
        window_end = next_available_date + timedelta(days=window_days)

        # Step 3: trades within window
        candidates = future_trades[future_trades['Entry_Date'] <= window_end]

        # If no trades in window → fallback to earliest few trades
        if candidates.empty:
            candidates = future_trades.head(5)

        # Step 4: random pick within candidates
        trade = candidates.sample(n=1).iloc[0]

        entry_date = trade['Entry_Date']
        ticker = trade['Ticker']
        profit = trade['Profit']

        # Apply return
        capital *= (1 + profit)

        exit_date = entry_date + timedelta(days=max_holding_days)

        results.append({
            'Entry_Date': entry_date,
            'Exit_Date': exit_date,
            'Ticker': ticker,
            'Profit': profit,
            'Capital': capital
        })

        # Move forward
        next_available_date = exit_date

    return pd.DataFrame(results)

In [ ]:
results.columns

In [ ]:
results['Entry_Date'] = pd.to_datetime(results['Entry_Date'])

In [ ]:
investment = investment_random_near(results)

In [ ]:
investment

In [ ]:
investment['Capital'].plot(logy=True)

In [ ]:
results_after_2025 = results[results['Entry_Date'] >= '2025-01-01']

In [ ]:
investment_after_2025 = investment_random_near(results_after_2025)

In [ ]:
investment_after_2025['Capital'].plot()

In [ ]:
results_after_2026 = results[results['Entry_Date'] >= '2026-01-01']
investment_after_2026 = investment_random_near(results_after_2026)

In [ ]:
investment_after_2026['Capital'].plot()

In [ ]:
risk_screened = results[results['Risk'] < 0.081]

In [ ]:
risk_screened_investment = investment_random_near(risk_screened)

In [ ]:
risk_screened_investment['Capital'].plot(logy=True)

# Step 4. Daily Top-N Earnings Signals Scanner

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import yfinance as yf
from datetime import datetime, timedelta
import math
from tqdm.auto import tqdm
import requests
import matplotlib.pyplot as plt

In [ ]:
# Helper files and parameters
# set parameters
START_DATE_1y = (datetime.today() - timedelta(days=180)).strftime('%Y-%m-%d') # half year history
END_DATE = datetime.today().strftime('%Y-%m-%d')

# download S&P500 tickers and their prices
def get_sp500_tickers():
    url = "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies"
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
    }
    response = requests.get(url, headers=headers)
    response.raise_for_status() # Raise an exception for HTTP errors

    # pandas can directly parse tables from HTML content
    tables = pd.read_html(response.text)

    df = None
    # Iterate through found tables to identify the S%P 500 constituents table
    for df_candidate in tables:
        # Check for columns commonly present in the S%P 500 constituents table
        # like 'Symbol', 'Ticker', or 'symbol' (case-insensitive)
        # Convert column names to string before comparison
        cols = [c for c in df_candidate.columns if "Symbol" in str(c) or "Ticker" in str(c) or "symbol" in str(c).lower()]
        if cols:
            df = df_candidate
            break # Found the table

    if df is None:
        raise ValueError("Could not find S%26P 500 constituents table on the Wikipedia page.")

    # Ensure 'Symbol' or 'Ticker' column is correctly identified
    # Convert column names to string before comparison
    col_name = [c for c in df.columns if "Symbol" in str(c) or "Ticker" in str(c) or "symbol" in str(c).lower()][0]

    # Clean up ticker symbols (e.g., BRK.B -> BRK-B)
    tickers = df[col_name].astype(str).str.replace(".", "-", regex=False).tolist()
    return tickers

def download_stock_prices(tickers_to_download, start, end, batch_size=80):
    """
    Download full OHLCV price data in batches using yfinance.
    Returns a MultiIndex DataFrame:
        columns = (ticker, price_field)
        index   = dates
    """
    all_data = pd.DataFrame()
    tickers_list = list(tickers_to_download)

    for i in tqdm(range(0, len(tickers_list), batch_size), desc="Downloading prices"):
        batch = tickers_list[i:i+batch_size]

        try:
            data = yf.download(
                batch, start=start, end=end,
                progress=False, group_by='ticker', auto_adjust=True
            )

            if data.empty:
                print(f"Skipping batch {batch}: No data downloaded.")
                continue

            # If single ticker, YF returns normal DataFrame → convert it to MultiIndex
            if not isinstance(data.columns, pd.MultiIndex):
                t = batch[0]
                data.columns = pd.MultiIndex.from_product([[t], data.columns])

            # Merge batch with accumulated data
            if all_data.empty:
                all_data = data
            else:
                all_data = pd.concat([all_data, data], axis=1)

        except Exception as e:
            print(f"Batch download error for {batch}: {e}")

    # Ensure sorted index and columns
    if not all_data.empty:
        all_data = all_data.sort_index()
        all_data = all_data.sort_index(axis=1)

    return all_data

# earnings event detection
def detect_event(price):

    gap = (
        price["Open"].iloc[-1] -
        price["Close"].iloc[-2]
    ) / price["Close"].iloc[-2]

    vol_ratio = (
        price["Volume"].iloc[-1] /
        price["Volume"].iloc[-50:].mean()
    )

    if abs(gap) > 0.1 and vol_ratio > 2:
        return True, gap, vol_ratio # Return all 3 values when True

    return False, gap, vol_ratio # Return all 3 values even when False

# Earnings jump scanner
def advanced_earnings_scan(df,lookback_days):

    signals = []

    # Get the actual unique tickers present in the first level of the MultiIndex
    actual_tickers = df.columns.get_level_values(0).unique()

    for ticker in actual_tickers:

        price = df[ticker].dropna()

        if len(price) < 120:
            continue

        # check last N days
        for i in range(lookback_days, 0, -1):
             hist = price.iloc[:-i]

             # Ensure 'hist' has enough data points for detect_event
             if len(hist) < 2:
                 continue

            # Event detection
             event_ok, gap, vol_ratio = detect_event(hist)

             if not event_ok:
                continue

             event_price = hist["Close"].iloc[-1]
             event_date = hist.index[-1]
             event_volume = hist["Volume"].iloc[-1]
             event_jump = gap

             # Calculate
             regime = hist["regime"].iloc[-1]

             signals.append({
                "Ticker": ticker,
                "Event_Date": event_date,
                "Event_Price": event_price,
                "Event_Volume": event_volume,
                "Event_Jump": event_jump,
                "regime": regime
            })

    signals = pd.DataFrame(signals)

    if len(signals) == 0:
        # print("No earnings events in the last", lookback_days, "days")
        return signals

    signals = signals.sort_values("Event_Date", ascending=False)

    return signals

def match_earnings(jump_date, earnings_date):
  return jump_date == earnings_date or jump_date == earnings_date + pd.Timedelta(days=1)

def check_consolidation(price, event_date):

    post_df = price.loc[event_date : event_date + pd.Timedelta(days=10)].copy() # check following 2 weeks after earnings

    if len(post_df) < 5:
        return 0

    score = 0

    # Rule 1
    peak = post_df['High'].iloc[0]
    trough = post_df['Low'].min()
    if (peak - trough) / peak <= 0.25:
        score += 1

    # Rule 2
    post_df['range'] = post_df['High'] - post_df['Low']
    avg_range = post_df['range'].mean()
    red_days = post_df[post_df['Close'] < post_df['Open']]
    if len(red_days) > 0 and (red_days['range'] < avg_range).mean() > 0.7:
        score += 1

    # Rule 3
    ma20 = price['Close'].rolling(20).mean()
    post_df['ma20'] = ma20.loc[post_df.index]
    if (post_df['Low'] > post_df['ma20'] * 0.98).mean() > 0.7:
        score += 1

    # Rule 4
    valid_length = len(post_df)
    if valid_length >= 5:
      score += 1

    # Rule 5 (vol contraction)
    vol = post_df['range'] / post_df['Close']
    if vol.iloc[-3:].mean() <= vol.mean():
        score += 1

    # Rule 6 (tight near highs)
    total_range = (post_df['High'].max() - post_df['Low'].min()) / post_df['Close'].iloc[-1]
    near_high = post_df['Close'].iloc[-1] > post_df['High'].max() * 0.9

    if total_range < 0.1 and near_high:
        score += 1

    return score


# based on score system
def find_entry(price, event_date, lookforward=10, max_entry_days=15):
    price = price.sort_index()

    if event_date not in price.index:
        return None, 0

    event_idx = price.index.get_loc(event_date)

    # consolidation window
    post_df = price.iloc[event_idx : event_idx + lookforward]

    if len(post_df) < 5:
        return None, 0

    # softer pivot
    pivot = post_df['High'].rolling(3).max().iloc[-1]

    # entry window
    future_df = price.iloc[event_idx + lookforward : event_idx + max_entry_days].copy()

    if len(future_df) == 0:
        return None, 0

    future_df['vol_avg'] = future_df['Volume'].rolling(20).mean()

    # scoring system
    for i in range(len(future_df)):
        row = future_df.iloc[i]

        score = 0

        # Rule 1: intraday breakout (looser)
        if row['High'] > pivot:
            score += 2

        # Rule 2: close strength
        if row['Close'] > pivot * 0.98:
            score += 1

        # Rule 3: volume confirmation (relaxed)
        if row['Volume'] > 1.2 * row['vol_avg']:
            score += 1

        # Rule 4: strong close in range
        if (row['Close'] - row['Low']) / (row['High'] - row['Low'] + 1e-6) > 0.6:
            score += 1

        # Entry threshold
        if score >= 3:
            return future_df.index[i], score

    return None, 0


In [ ]:
# Define MIN_PRICE_HISTORY_DAYS before it's used
MIN_PRICE_HISTORY_DAYS = 120 # at least more than 4 months

# Fetch tickers and prices
sp500_tickers = get_sp500_tickers()
universe = [t for t in sp500_tickers]

prices_1y = download_stock_prices(universe, start=START_DATE_1y, end=END_DATE)
# Remove tickers with insufficient history
valid = [t for t in sp500_tickers if prices_1y.get(t, pd.Series()).dropna().shape[0] >= MIN_PRICE_HISTORY_DAYS]
print(f"{len(valid)} tickers with >= {MIN_PRICE_HISTORY_DAYS} days of data.")

data = prices_1y.copy()

# prepare S&P 500 index data, don't keet ticker's name

spx = yf.download("^GSPC", start=START_DATE_1y, end=END_DATE)

spx.columns = spx.columns.droplevel(1)

spx

def label_regime(spy):

    close = spy["Close"]
    ma200 = close.rolling(200).mean()

    ret3 = close.pct_change(63)

    regime = []

    for i in range(len(close)):

        if close.iloc[i] > ma200.iloc[i] and ret3.iloc[i] > 0:
            regime.append(1)   # bull
        elif close.iloc[i] < ma200.iloc[i]:
            regime.append(-1)  # bear
        else:
            regime.append(0)   # neutral

    spy["regime"] = regime

    return spy

spx = label_regime(spx)

# combine data with spx['regime']
# First, align the spx['regime'] Series to the 'data' DataFrame's index.
# Using 'ffill' will propagate the last valid observation forward to fill any missing dates.
aligned_regime = spx['regime'].reindex(data.index, method='ffill')

# Iterate through each unique ticker and add the aligned 'regime' series
# as a new sub-column under that ticker in the MultiIndex DataFrame.
for ticker in data.columns.get_level_values(0).unique():
    data[ticker, 'regime'] = aligned_regime

signals = advanced_earnings_scan(data, 60) # only check last two months' events

calendar = pd.DataFrame()
for ticker_symbol in signals["Ticker"].unique():
  ticker = yf.Ticker(ticker_symbol)
  try:
    # Get earnings dates
    earnings = ticker.get_earnings_dates(limit=2) # change to 2. only need recent info

    # Check if earnings DataFrame is not empty before processing
    if not earnings.empty:
      # Assign the index (which contains the datetime) to a new 'date' column
      earnings['date'] = earnings.index
      # Convert the 'date' column to contain only the date part
      earnings['date'] = earnings['date'].dt.date
      earnings['Ticker'] = ticker_symbol

      calendar = pd.concat([calendar, earnings], ignore_index=True)
  except KeyError as e:
    print(f"Skipping ticker {ticker_symbol} due to KeyError: {e} in get_earnings_dates.")
  except Exception as e:
    print(f"Skipping ticker {ticker_symbol} due to unexpected error: {e} in get_earnings_dates.")

signals['Event_Date'] = pd.to_datetime(signals['Event_Date'])
calendar['date'] = pd.to_datetime(calendar['date'])

# better method, avoid the O(N x M) nested loops, use merge_asof
true_signals = pd.merge_asof(
    signals.sort_values('Event_Date'),
    calendar.sort_values('date'),
    left_on='Event_Date',
    right_on='date',
    by='Ticker',
    direction='nearest',
    tolerance=pd.Timedelta('1D')
)

# drop off NaN data on date
true_signals = true_signals.dropna(subset=['date'])

true_signals['Event_Date'] = pd.to_datetime(true_signals['Event_Date'])

true_signals['Consolidation_Score'] = 0
true_signals['Entry_Date'] = None

grouped = true_signals.groupby('Ticker')

for ticker, group in grouped:
    price = data.get(ticker)

    if price is None:
        continue

    price = price.sort_index()

    for idx, row in group.iterrows():
        event_date = row['Event_Date']

        try:
            score = check_consolidation(price, event_date)
            entry, signal_score = find_entry(price, event_date)
        except:
            score = 0
            entry = None
            signal_score = 0

        true_signals.loc[idx, 'Consolidation_Score'] = score
        true_signals.loc[idx, 'Entry_Date'] = entry.date() if entry is not None else None
        true_signals.loc[idx, 'Entry_Price'] = price.loc[entry, 'Close'] if entry is not None else None
        true_signals.loc[idx, 'Signal_Score'] = signal_score

entrys = true_signals.dropna(subset=['Entry_Date'])

entrys['Action'] = np.where(entrys['Surprise(%)'] > 0, 1, -1)

# arbitrarily set Risk at 0.08
entrys['Stop'] = np.where(entrys['Action'] ==1, entrys['Entry_Price'] * (1 - 0.08), entrys['Entry_Price'] * (1 + 0.08))
cols = ['Ticker', 'Event_Date', 'Event_Jump',
       'regime', 'Surprise(%)', 'Consolidation_Score', 'Entry_Date', 'Entry_Price', 'Signal_Score',
       'Action', 'Stop']
final_entrys = entrys[cols]
print(final_entrys)

In [ ]:
final_entrys.sort_values('Entry_Date', ascending=False)

# what if we change to use Nasdaq tickers?


# other try, based on Event_Jump

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import yfinance as yf
from datetime import datetime, timedelta
import math
from tqdm.auto import tqdm
import requests
import matplotlib.pyplot as plt

In [ ]:
# set parameters
START_DATE = (datetime.today() - timedelta(days=365*20)).strftime('%Y-%m-%d') # 5y of history // change to 20 years
END_DATE = datetime.today().strftime('%Y-%m-%d')

# download S&P500 tickers and their prices
def get_nasdaq100_tickers():
    url = "https://en.wikipedia.org/wiki/NASDAQ-100"
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
    }
    response = requests.get(url, headers=headers)
    response.raise_for_status() # Raise an exception for HTTP errors

    # pandas can directly parse tables from HTML content
    tables = pd.read_html(response.text)

    df = None
    # Iterate through found tables to identify the S%P 500 constituents table
    for df_candidate in tables:
        # Check for columns commonly present in the S%P 500 constituents table
        # like 'Symbol', 'Ticker', or 'symbol' (case-insensitive)
        # Convert column names to string before comparison
        cols = [c for c in df_candidate.columns if "Symbol" in str(c) or "Ticker" in str(c) or "symbol" in str(c).lower()]
        if cols:
            df = df_candidate
            break # Found the table

    if df is None:
        raise ValueError("Could not find Nasdaq constituents table on the Wikipedia page.")

    # Ensure 'Symbol' or 'Ticker' column is correctly identified
    # Convert column names to string before comparison
    col_name = [c for c in df.columns if "Symbol" in str(c) or "Ticker" in str(c) or "symbol" in str(c).lower()][0]

    # Clean up ticker symbols (e.g., BRK.B -> BRK-B)
    tickers = df[col_name].astype(str).str.replace(".", "-", regex=False).tolist()
    return tickers


def download_stock_prices(tickers_to_download, start, end, batch_size=80):
    """
    Download full OHLCV price data in batches using yfinance.
    Returns a MultiIndex DataFrame:
        columns = (ticker, price_field)
        index   = dates
    """
    all_data = pd.DataFrame()
    tickers_list = list(tickers_to_download)

    for i in tqdm(range(0, len(tickers_list), batch_size), desc="Downloading prices"):
        batch = tickers_list[i:i+batch_size]

        try:
            data = yf.download(
                batch, start=start, end=end,
                progress=False, group_by='ticker', auto_adjust=True
            )

            if data.empty:
                print(f"Skipping batch {batch}: No data downloaded.")
                continue

            # If single ticker, YF returns normal DataFrame → convert it to MultiIndex
            if not isinstance(data.columns, pd.MultiIndex):
                t = batch[0]
                data.columns = pd.MultiIndex.from_product([[t], data.columns])

            # Merge batch with accumulated data
            if all_data.empty:
                all_data = data
            else:
                all_data = pd.concat([all_data, data], axis=1)

        except Exception as e:
            print(f"Batch download error for {batch}: {e}")

    # Ensure sorted index and columns
    if not all_data.empty:
        all_data = all_data.sort_index()
        all_data = all_data.sort_index(axis=1)

    return all_data

# Step 1. download the stock price and find the event signals

In [ ]:
# Define MIN_PRICE_HISTORY_DAYS before it's used
MIN_PRICE_HISTORY_DAYS = 252 # Approximately 1 year of trading days

# Fetch tickers and prices
nasdaq_tickers = get_nasdaq100_tickers()
universe = [t for t in nasdaq_tickers]  # copy

prices = download_stock_prices(universe, start=START_DATE, end=END_DATE)
# Remove tickers with insufficient history
valid = [t for t in nasdaq_tickers if prices.get(t, pd.Series()).dropna().shape[0] >= MIN_PRICE_HISTORY_DAYS]
print(f"{len(valid)} tickers with >= {MIN_PRICE_HISTORY_DAYS} days of data.")

In [ ]:
data = prices.copy()

In [ ]:
# mount on google drive and save the data
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# save the prices data
prices.to_csv('/content/drive/MyDrive/ML/nasdaq_prices_20y.csv')

In [ ]:
data = pd.read_csv('/content/drive/MyDrive/ML/nasdaq_prices_20y.csv', header=[0,1], index_col=0)
data.index = pd.to_datetime(data.index)

In [ ]:
# prepare S&P 500 index data, don't keet ticker's name

spx = yf.download("^GSPC", start=START_DATE, end=END_DATE)

spx.columns = spx.columns.droplevel(1)

spx

def label_regime(spy):

    close = spy["Close"]
    ma200 = close.rolling(200).mean()

    ret3 = close.pct_change(63)

    regime = []

    for i in range(len(close)):

        if close.iloc[i] > ma200.iloc[i] and ret3.iloc[i] > 0:
            regime.append(1)   # bull
        elif close.iloc[i] < ma200.iloc[i]:
            regime.append(-1)  # bear
        else:
            regime.append(0)   # neutral

    spy["regime"] = regime

    return spy

spx = label_regime(spx)

# combine data with spx['regime']
# First, align the spx['regime'] Series to the 'data' DataFrame's index.
# Using 'ffill' will propagate the last valid observation forward to fill any missing dates.
aligned_regime = spx['regime'].reindex(data.index, method='ffill')

# Iterate through each unique ticker and add the aligned 'regime' series
# as a new sub-column under that ticker in the MultiIndex DataFrame.
for ticker in data.columns.get_level_values(0).unique():
    data[ticker, 'regime'] = aligned_regime

In [ ]:
# earnings event detection
def detect_event(price):

    gap = (
        price["Open"].iloc[-1] -
        price["Close"].iloc[-2]
    ) / price["Close"].iloc[-2]

    vol_ratio = (
        price["Volume"].iloc[-1] /
        price["Volume"].iloc[-50:].mean()
    )

    if abs(gap) > 0.1 and vol_ratio > 2:
        return True, gap, vol_ratio # Return all 3 values when True

    return False, gap, vol_ratio # Return all 3 values even when False

In [ ]:
# Earnings jump scanner
def advanced_earnings_scan(df,lookback_days):

    signals = []

    # Get the actual unique tickers present in the first level of the MultiIndex
    actual_tickers = df.columns.get_level_values(0).unique()

    for ticker in actual_tickers:

        price = df[ticker].dropna()

        if len(price) < 150:
            continue

        # check last N days
        for i in range(lookback_days, 0, -1):
             hist = price.iloc[:-i]

             # Ensure 'hist' has enough data points for detect_event
             if len(hist) < 2:
                 continue

            # Event detection
             event_ok, gap, vol_ratio = detect_event(hist)

             if not event_ok:
                continue

             event_price = hist["Close"].iloc[-1]
             event_date = hist.index[-1]
             event_volume = hist["Volume"].iloc[-1]
             event_jump = gap

             # Calculate
             regime = hist["regime"].iloc[-1]

             signals.append({
                "Ticker": ticker,
                "Event_Date": event_date,
                "Event_Price": event_price,
                "Event_Volume": event_volume,
                "Event_Jump": event_jump,
                "regime": regime
            })

    signals = pd.DataFrame(signals)

    if len(signals) == 0:
        # print("No earnings events in the last", lookback_days, "days")
        return signals

    signals = signals.sort_values("Event_Date", ascending=False)

    return signals

signals = advanced_earnings_scan(data, len(data))

In [ ]:
calendar = pd.DataFrame()
for ticker_symbol in signals["Ticker"].unique():
  ticker = yf.Ticker(ticker_symbol)

  # Get earnings dates
  earnings = ticker.get_earnings_dates(limit=80)

  # Check if earnings DataFrame is not empty before processing
  if not earnings.empty:
    # Assign the index (which contains the datetime) to a new 'date' column
    earnings['date'] = earnings.index
    # Convert the 'date' column to contain only the date part
    earnings['date'] = earnings['date'].dt.date
    earnings['Ticker'] = ticker_symbol

    calendar = pd.concat([calendar, earnings], ignore_index=True)

In [ ]:
len(calendar),len(signals)

In [ ]:
signals['Event_Date'] = pd.to_datetime(signals['Event_Date'])
calendar['date'] = pd.to_datetime(calendar['date'])

In [ ]:
# better method, avoid the O(N x M) nested loops, use merge_asof
true_signals = pd.merge_asof(
    signals.sort_values('Event_Date'),
    calendar.sort_values('date'),
    left_on='Event_Date',
    right_on='date',
    by='Ticker',
    direction='nearest',
    tolerance=pd.Timedelta('1D')
)

In [ ]:
true_signals

In [ ]:
# drop off NaN data on date
true_signals = true_signals.dropna(subset=['date'])

In [ ]:
true_signals

# Step 2. the consolidation analysis

Earnings Jump → Pullback → Tight Consolidation → Breakout Entry

In [ ]:
def check_consolidation(price, event_date):

    post_df = price.loc[event_date : event_date + pd.Timedelta(days=10)].copy() # check following 2 weeks after earnings

    if len(post_df) < 5:
        return 0

    score = 0

    # Rule 1
    peak = post_df['High'].iloc[0]
    trough = post_df['Low'].min()
    if (peak - trough) / peak <= 0.25:
        score += 1

    # Rule 2
    post_df['range'] = post_df['High'] - post_df['Low']
    avg_range = post_df['range'].mean()
    red_days = post_df[post_df['Close'] < post_df['Open']]
    if len(red_days) > 0 and (red_days['range'] < avg_range).mean() > 0.7:
        score += 1

    # Rule 3
    ma20 = price['Close'].rolling(20).mean()
    post_df['ma20'] = ma20.loc[post_df.index]
    if (post_df['Low'] > post_df['ma20'] * 0.98).mean() > 0.7:
        score += 1

    # Rule 4
    valid_length = len(post_df)
    if valid_length >= 5:
      score += 1

    # Rule 5 (vol contraction)
    vol = post_df['range'] / post_df['Close']
    if vol.iloc[-3:].mean() <= vol.mean():
        score += 1

    # Rule 6 (tight near highs)
    total_range = (post_df['High'].max() - post_df['Low'].min()) / post_df['Close'].iloc[-1]
    near_high = post_df['Close'].iloc[-1] > post_df['High'].max() * 0.9

    if total_range < 0.1 and near_high:
        score += 1

    return score

In [ ]:
# based on score system
def find_entry(price, event_date, lookforward=10, max_entry_days=15):
    price = price.sort_index()

    if event_date not in price.index:
        return None, 0

    event_idx = price.index.get_loc(event_date)

    # consolidation window
    post_df = price.iloc[event_idx : event_idx + lookforward]

    if len(post_df) < 5:
        return None, 0

    # softer pivot
    pivot = post_df['High'].rolling(3).max().iloc[-1]

    # entry window
    future_df = price.iloc[event_idx + lookforward : event_idx + max_entry_days].copy()

    if len(future_df) == 0:
        return None, 0

    future_df['vol_avg'] = future_df['Volume'].rolling(20).mean()

    # scoring system
    for i in range(len(future_df)):
        row = future_df.iloc[i]

        score = 0

        # Rule 1: intraday breakout (looser)
        if row['High'] > pivot:
            score += 2

        # Rule 2: close strength
        if row['Close'] > pivot * 0.98:
            score += 1

        # Rule 3: volume confirmation (relaxed)
        if row['Volume'] > 1.2 * row['vol_avg']:
            score += 1

        # Rule 4: strong close in range
        if (row['Close'] - row['Low']) / (row['High'] - row['Low'] + 1e-6) > 0.6:
            score += 1

        # Entry threshold
        if score >= 3:
            return future_df.index[i], score

    return None, 0

In [ ]:
true_signals['Event_Date'] = pd.to_datetime(true_signals['Event_Date'])

true_signals['Consolidation_Score'] = 0
true_signals['Entry_Date'] = None

grouped = true_signals.groupby('Ticker')

for ticker, group in grouped:
    price = data.get(ticker)

    if price is None:
        continue

    price = price.sort_index()

    for idx, row in group.iterrows():
        event_date = row['Event_Date']

        try:
            score = check_consolidation(price, event_date)
            entry, signal_score = find_entry(price, event_date)
        except:
            score = 0
            entry = None
            signal_score = 0

        true_signals.loc[idx, 'Consolidation_Score'] = score
        true_signals.loc[idx, 'Entry_Date'] = entry.date() if entry is not None else None
        true_signals.loc[idx, 'Entry_Price'] = price.loc[entry, 'Close'] if entry is not None else None
        true_signals.loc[idx, 'Signal_Score'] = signal_score

In [ ]:
entrys = true_signals.dropna(subset=['Entry_Date'])

In [ ]:
len(entrys)

In [ ]:
entrys['Action'] = np.where(entrys['Surprise(%)'] > 0, 1, -1)

In [ ]:
# arbitarily set Risk at 0.08
entrys['Stop'] = np.where(entrys['Action'] ==1, entrys['Entry_Price'] * (1 - 0.08), entrys['Entry_Price'] * (1 + 0.08))

# Step 3. backtest the entries

In [ ]:
def calculate_profit(price, action, entry_date, entry_price, stop_price, max_holding_days=40):
  """
    Returns:
        profit,
        lowest price,
        highest price,
        days_to_peak,
        days_to_bottom,
        risk
    """

  # 0. start after entry day
  trade = price.loc[entry_date:].iloc[1:].copy()

  if len(trade) <= max_holding_days:
    return 0,0,0,0,0,0

  # 1. Apply holding cap
  trade = trade.iloc[:max_holding_days]

  # 2. Find lowest / highest price and its date
  max_price = trade["Close"].max()
  max_idx = trade["Close"].idxmax()

  min_price = trade["Close"].min()
  min_idx = trade["Close"].idxmin()

  # 4. Compute days to peak
  days_to_peak = trade.index.get_loc(max_idx) + 1
  days_to_bottom = trade.index.get_loc(min_idx) + 1

  # Profit
  if action > 0:
    profit = (max_price - entry_price) / entry_price
    # Initialize risk with the stop-loss distance
    risk = abs(stop_price - entry_price) / entry_price
    # If the price dropped further than the stop before the peak, that's the actual risk faced
    if min_price < stop_price and days_to_bottom < days_to_peak:
        profit = (stop_price - entry_price) / entry_price
        risk = abs(min_price - entry_price) / entry_price

  elif action < 0:
    profit = (entry_price - min_price) / entry_price
    # Initialize risk with the stop-loss distance
    risk = abs(stop_price - entry_price) / entry_price
    # If the price rose further than the stop (for short) before the bottom, that's the actual risk faced
    if max_price > stop_price and days_to_bottom > days_to_peak:
        profit = (entry_price - stop_price) / entry_price
        risk = abs(max_price - entry_price) / entry_price

  return profit, risk, days_to_peak, days_to_bottom, max_price, min_price

In [ ]:
entrys.head()

In [ ]:
results = []

for _, row in entrys.iterrows():

    ticker = row["Ticker"]
    entry_date = row["Entry_Date"]
    entry_price = row["Entry_Price"]
    stop_price = row["Stop"]
    action = row["Action"]

    price = data[ticker].dropna()

    profit, risk, days_to_peak, days_to_bottom, max_price, min_price = calculate_profit(
        price,
        action,
        entry_date,
        entry_price,
        stop_price,
        max_holding_days=40
    )

    results.append({
        **row,
        "Profit": round(profit, 4),
        "Risk": round(risk, 4),
        "Days_to_Peak": days_to_peak,
        "Days_to_Bottom": days_to_bottom,
        "lowest_price": min_price,
        "highest_price": max_price,
        # "Label
        # "Label": label,
        # "Time_to_Outcome": time_to_outcome
    })

results = pd.DataFrame(results)

In [ ]:
results

In [ ]:
# does signal score correlates with profit
results[["Signal_Score", "Profit"]].corr()

In [ ]:
# scatter plot the correlations
import seaborn as sns
import matplotlib.pyplot as plt
plt.figure(figsize=(10, 7))
sns.scatterplot(
    data=results,
    x='Signal_Score',
    y='Profit',
    hue='Action',
    palette={1: 'green', -1: 'red'}
)

In [ ]:
# does there is a threshold to increase the win rate?
# first to see the average or median Signal_Score
results.groupby("Action")["Signal_Score"].mean()

In [ ]:
results.groupby("Action")["Signal_Score"].median()

In [ ]:
# second to see the win rate for each action
results.groupby("Action")["Profit"].apply(lambda x: (x > 0).mean())

In [ ]:
results.groupby("Action")["Profit"].sum()

In [ ]:
results[results["Profit"] > 0]["Risk"].mean(), results[results["Profit"] > 0]["Risk"].median()

In [ ]:
results[results["Profit"] < 0]["Risk"].mean(), results[results["Profit"] < 0]["Risk"].median()

In [ ]:
results["Risk"].describe()

In [ ]:
results[results['Profit'] > 0]["Risk"].describe()

In [ ]:
profit_risk = results[(results['Risk'] < 0.081) & (results['Profit'] > 0 )].groupby("Action")["Profit"].sum()
loss_risk = results[(results['Risk'] < 0.081) & (results['Profit'] < 0 )].groupby("Action")["Profit"].sum()

In [ ]:
profit_loss_risk = pd.DataFrame([profit_risk, loss_risk]).T
profit_loss_risk.columns = ['Profit', 'Loss']
profit_loss_risk['ratio'] = abs(profit_loss_risk['Profit'] / profit_loss_risk['Loss'])

In [ ]:
profit_loss_risk

In [ ]:
# Action × Profit Distribution
results.boxplot(column="Profit", by="Action")

In [ ]:
results[results['Risk'] <= 0.081].boxplot(column="Profit", by="Action")

In [ ]:
# Expectancy
win = results[results["Profit"] > 0].groupby("Action")["Profit"].mean()
loss = results[results["Profit"] <= 0].groupby("Action")["Profit"].mean()
# Calculate win_rate as the proportion of trades with profit > 0 for each action
win_rate = results.groupby("Action")["Profit"].apply(lambda x: (x > 0).mean())

expectancy = win_rate * win + (1 - win_rate) * loss

In [ ]:
expectancy

In [ ]:
# if using 0.081 as Risk cut, what is the average days to the maximum price
results[results['Risk'] < 0.081]['Days_to_Peak'].mean(), results[results['Risk'] < 0.081]['Days_to_Peak'].median()

In [ ]:
results[results['Risk'] < 0.081]['Days_to_Bottom'].mean(), results[results['Risk'] < 0.081]['Days_to_Bottom'].median()

In [ ]:
results['Profit'].sum()

In [ ]:
results[results['Risk'] < 0.081]['Profit'].sum()

In [ ]:
stock_profit = results.groupby('Ticker')['Profit'].sum()


In [ ]:
stock_profit.sort_values(ascending=False).plot(kind='bar')

## Random choose test

start with $10,000. randomly chose a entry and exit as mentioned in results. To see the real-time performance

In [ ]:
import pandas as pd
import random
from datetime import timedelta

def investment_random_near(df, initial=10000, max_holding_days=40, window_days=5):
    df = df.sort_values("Entry_Date").reset_index(drop=True)

    capital = initial
    next_available_date = df['Entry_Date'].min()

    results = []

    while True:
        # Step 1: all future trades
        future_trades = df[df['Entry_Date'] >= next_available_date]

        if future_trades.empty:
            break

        # Step 2: define time window
        window_end = next_available_date + timedelta(days=window_days)

        # Step 3: trades within window
        candidates = future_trades[future_trades['Entry_Date'] <= window_end]

        # If no trades in window → fallback to earliest few trades
        if candidates.empty:
            candidates = future_trades.head(5)

        # Step 4: random pick within candidates
        trade = candidates.sample(n=1).iloc[0]

        entry_date = trade['Entry_Date']
        ticker = trade['Ticker']
        profit = trade['Profit']

        # Apply return
        capital *= (1 + profit)

        exit_date = entry_date + timedelta(days=max_holding_days)

        results.append({
            'Entry_Date': entry_date,
            'Exit_Date': exit_date,
            'Ticker': ticker,
            'Profit': profit,
            'Capital': capital
        })

        # Move forward
        next_available_date = exit_date

    return pd.DataFrame(results)

In [ ]:
results.columns

In [ ]:
results['Entry_Date'] = pd.to_datetime(results['Entry_Date'])

In [ ]:
investment = investment_random_near(results)

In [ ]:
investment

In [ ]:
investment['Capital'].plot(logy=True)

In [ ]:
results_after_2025 = results[results['Entry_Date'] >= '2025-01-01']

In [ ]:
investment_after_2025 = investment_random_near(results_after_2025)

In [ ]:
investment_after_2025['Capital'].plot()

In [ ]:
results_2024 = results[(results['Entry_Date'] >= '2024-01-01') & (results['Entry_Date'] < '2025-01-01')]

In [ ]:
investment_2024 = investment_random_near(results_2024)

In [ ]:
investment_2024['Capital'].plot()

In [ ]:
investment_2024